# Phase 3: Submit RFT Job for o4-mini

This notebook submits a Reinforcement Fine-Tuning (RFT) job on Azure AI Foundry for the o4-mini model using the Zava-Next agent task.

## What This Notebook Does

1. Loads training data (`rft_next_train_v2.jsonl` - 192 scenarios)
2. Loads validation data (`rft_next_val_v2.jsonl` - 62 scenarios)
3. Uploads datasets to Azure AI Foundry
4. Configures the custom Python grader (zava_quality)
5. Configures tool access (6 tools for the agent)
6. Submits the RFT job with proper hyperparameters
7. Monitors job progress

## Prerequisites

- `.env` file configured with `PROJECT_ENDPOINT` and `TOOL_URL`
- Tool endpoint deployed and accessible (from Phase 1)
- Training data fixed (`rft_next_train_v2.jsonl`)
- Validation data fixed (`rft_next_val_v2.jsonl`)

## Expected Runtime

- **Submission:** ~2-3 minutes
- **Training:** 2-4 hours (depends on compute availability)
- **Total:** ~2.5-4.5 hours

## Reference Run

- Previous successful run: `o4-mini-2025-04-16.ft-1839d286a0654e459697cd02cb8eb9a4-zava-next-rft`
- Resource: `omi-build-demo-ncus`
- This notebook uses similar configuration

In [1]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

# Load environment
load_dotenv()

PROJECT_ENDPOINT = os.getenv('PROJECT_ENDPOINT')
TOOL_URL = os.getenv('TOOL_URL')  # Your deployed function app URL

if not PROJECT_ENDPOINT:
    raise ValueError("PROJECT_ENDPOINT not found in .env file")

if not TOOL_URL:
    raise ValueError(
        "TOOL_URL not found in .env file.\n"
        "Please add: TOOL_URL=https://your-function-app.azurewebsites.net/api"
    )

# Initialize clients
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project_client.get_openai_client()

print(f"✅ Connected to Azure AI Foundry")
print(f"   Project: {PROJECT_ENDPOINT}")
print(f"   Tool URL: {TOOL_URL}")

✅ Connected to Azure AI Foundry
   Project: https://omi-ignite-demo-resource.services.ai.azure.com/api/projects/omi-ignite-demo   #this is sweden
   Tool URL: https://zava-next-tools-omkarm.azurewebsites.net


## 1. Load Training and Validation Data

We use:
- **Training:** 192 scenarios (181 action + 11 clarification)
- **Validation:** 62 scenarios (52 action + 10 clarification)

Both datasets have been fixed to handle scenarios with missing order IDs.

In [2]:
# Paths
train_path = "../data/rft_next_train_v2.jsonl"
val_path = "../data/rft_next_val_v2.jsonl"

# Load data
with open(train_path) as f:
    train_data = [json.loads(line) for line in f]

with open(val_path) as f:
    val_data = [json.loads(line) for line in f]

print(f"✅ Loaded training data: {len(train_data)} scenarios")
print(f"✅ Loaded validation data: {len(val_data)} scenarios")

# Preview format
sample = train_data[0]
print(f"\nSample scenario keys: {list(sample.keys())}")
print(f"\nSample user message:")
print(f'  "{sample["messages"][1]["content"][:100]}..."')
print(f"\nExpected resolution:")
print(f'  "{sample["expected_resolution"][:100]}..."')

✅ Loaded training data: 192 scenarios
✅ Loaded validation data: 62 scenarios

Sample scenario keys: ['messages', 'expected_resolution', 'expected_tools', 'expected_actions', 'expected_amounts', 'order_id', 'target_items', 'difficulty', 'scenario_id']

Sample user message:
  "Yusuf Rossi. Need to exchange the Canvas Tote Bag from ORD-012 for a different size...."

Expected resolution:
  "Action: exchange for LI-016 (reason: exchange). Amount: $0.00...."


## 2. Upload Datasets to Azure AI Foundry

Upload both training and validation datasets.

In [3]:
print("Uploading training dataset...")
train_file = client.files.create(
    file=open(train_path, "rb"),
    purpose="fine-tune"
)
print(f"✅ Training file uploaded: {train_file.id}")

print("\nUploading validation dataset...")
val_file = client.files.create(
    file=open(val_path, "rb"),
    purpose="fine-tune"
)
print(f"✅ Validation file uploaded: {val_file.id}")

print(f"\n📊 Ready for RFT job submission")
print(f"   Training: {len(train_data)} scenarios")
print(f"   Validation: {len(val_data)} scenarios")

Uploading training dataset...


NotFoundError: Error code: 404 - {'error': {'code': 'ResourceNotFound', 'message': 'The project does not exist.'}}

## 3. Load Custom Python Grader

The grader (`zava_grader_response.py`) scores agent responses on 3 dimensions:
- **Decision Correctness (50%)** - Right action?
- **Financial Accuracy (30%)** - Correct amounts?
- **Format Compliance (20%)** - Structured output?

The grader also handles clarification scenarios.

In [4]:
# Load the RFT grader
# This is a simpler grader designed specifically for RFT training
# (different data access patterns than Foundry evals)
with open("../eval/zava_grader_rft.py") as f:
    GRADER_SOURCE = f.read()

print("✅ RFT Grader loaded")
print(f"   File: eval/zava_grader_rft.py")
print(f"   Size: {len(GRADER_SOURCE)} characters")
print(f"\n   Scoring: 45% action + 35% amounts + 20% tools")
print(f"   Pass threshold: 0.80 (configured in job submission)")
print(f"\n   Based on successful run:")
print(f"   o4-mini-2025-04-16.ft-1839d286a0654e459697cd02cb8eb9a4-zava-next-rft")

✅ RFT Grader loaded
   File: eval/zava_grader_rft.py
   Size: 7822 characters

   Scoring: 45% action + 35% amounts + 20% tools
   Pass threshold: 0.80 (configured in job submission)

   Based on successful run:
   o4-mini-2025-04-16.ft-1839d286a0654e459697cd02cb8eb9a4-zava-next-rft


## 4. Configure Tool Access for RFT

The model needs access to 6 tools during training:
1. **get_order_details** - Retrieve order info, customer tier
2. **get_fulfillment_status** - Check delivery status
3. **check_resolution_policy** - Verify eligibility
4. **check_inventory** - Check stock for exchanges
5. **calculate_resolution** - Compute refund amounts
6. **submit_resolution** - Finalize the resolution

Each tool points to your deployed Function App endpoint.

In [5]:
# Tool configuration - all 6 tools from zava-next
#
# IMPORTANT: server_url should be the BASE URL only!
# Foundry will automatically append /{tool_name} when calling
#
TOOL_CONFIG = [
    {"name": "get_order_details", "server_url": f"{TOOL_URL}/tool", "headers": {}},
    {"name": "get_fulfillment_status", "server_url": f"{TOOL_URL}/tool", "headers": {}},
    {"name": "check_resolution_policy", "server_url": f"{TOOL_URL}/tool", "headers": {}},
    {"name": "check_inventory", "server_url": f"{TOOL_URL}/tool", "headers": {}},
    {"name": "calculate_resolution", "server_url": f"{TOOL_URL}/tool", "headers": {}},
    {"name": "submit_resolution", "server_url": f"{TOOL_URL}/tool", "headers": {}}
]

print("✅ Tool configuration ready:")
print(f"   Base URL: {TOOL_URL}/tool")
print(f"   Tools configured: {len(TOOL_CONFIG)}")
for i, tool in enumerate(TOOL_CONFIG, 1):
    print(f"   {i}. {tool['name']}")

print(f"\n💡 Foundry will call: {{server_url}}/{{tool_name}}")
print(f"   Example: {TOOL_URL}/tool/get_order_details")

✅ Tool configuration ready:
   Base URL: https://zava-next-tools-omkarm.azurewebsites.net/tool
   Tools configured: 6
   1. get_order_details
   2. get_fulfillment_status
   3. check_resolution_policy
   4. check_inventory
   5. calculate_resolution
   6. submit_resolution

💡 Foundry will call: {server_url}/{tool_name}
   Example: https://zava-next-tools-omkarm.azurewebsites.net/tool/get_order_details


## 5. Submit the RFT Job

Key parameters:
- **model:** `o4-mini` (reasoning model, ideal for agent tasks)
- **method:** `reinforcement` (not supervised)
- **pass_threshold:** 0.80 (grader score ≥ 0.80 counts as success)
- **max_episode_steps:** 5 (max tool-call rounds per scenario)
- **n_epochs:** 3 (training passes through data)
- **compute_multiplier:** 1.5 (extra compute for exploration)
- **reasoning_effort:** "medium" (o4-mini reasoning depth)

This configuration is based on the successful reference run:
`o4-mini-2025-04-16.ft-1839d286a0654e459697cd02cb8eb9a4-zava-next-rft`

In [6]:
# Generate unique suffix with timestamp
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
suffix = f"zava-next-rft-{timestamp}"

print("🚀 Submitting RFT job...")
print(f"   Base model: o4-mini")
print(f"   Suffix: {suffix}")
print(f"   Training: {len(train_data)} scenarios")
print(f"   Validation: {len(val_data)} scenarios")
print()

# Submit the job
job = client.fine_tuning.jobs.create(
    model="o4-mini",
    training_file=train_file.id,
    validation_file=val_file.id,
    suffix=suffix,
    method={
        "type": "reinforcement",
        "reinforcement": {
            "grader": {
                "type": "python",
                "name": "zava_quality",
                "source": GRADER_SOURCE.strip(),
                "pass_threshold": 0.80,
            },
            "tools": TOOL_CONFIG,
            "max_episode_steps": 5,
            "hyperparameters": {
                "n_epochs": 1,
                "learning_rate_multiplier": 1.0,
                "compute_multiplier": 1.5,
                "reasoning_effort": "medium",
                "eval_interval": 5,
                "eval_samples": 10,
            },
        }
    },
)

JOB_ID = job.id
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✅ RFT JOB SUBMITTED SUCCESSFULLY!")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"\nJob ID: {JOB_ID}")
print(f"Status: {job.status}")
print(f"Model: {job.model}")
print(f"Suffix: {suffix}")
print(f"\nFine-tuned model will be named:")
print(f"  {job.model}:<deployment-id>-{suffix}")
print(f"\n⏰ Expected completion time: 2-4 hours")
print(f"\n💡 Save this Job ID for monitoring!")

🚀 Submitting RFT job...
   Base model: o4-mini
   Suffix: zava-next-rft-20260527-032901
   Training: 192 scenarios
   Validation: 62 scenarios



NameError: name 'train_file' is not defined

## 6. Monitor Job Progress

Check the current status of your RFT job.

In [ ]:
# Check status
job_status = client.fine_tuning.jobs.retrieve(JOB_ID)

print(f"Job ID: {job_status.id}")
print(f"Status: {job_status.status}")
print(f"Model: {job_status.model}")
print(f"\nCreated: {job_status.created_at}")
print(f"Updated: {job_status.finished_at or 'In progress...'}")

if job_status.status == "succeeded":
    print(f"\n✅ JOB COMPLETED!")
    print(f"\nFine-tuned model ID: {job_status.fine_tuned_model}")
    print(f"\nNext steps:")
    print(f"  1. Deploy this model as a hosted agent")
    print(f"  2. Run evaluation (phase2_base_evaluations.ipynb)")
    print(f"  3. Compare with baseline o4-mini results")
elif job_status.status == "failed":
    print(f"\n❌ JOB FAILED")
    print(f"\nError: {job_status.error}")
else:
    print(f"\n⏳ Job is {job_status.status}...")
    print(f"   Check back in 30-60 minutes")
    print(f"\n   Or monitor in Azure AI Foundry portal:")
    print(f"   Go to Fine-tuning → Jobs → {JOB_ID}")

## 7. View Training Events (Optional)

Monitor detailed training progress and metrics.

In [ ]:
# List recent events
events = client.fine_tuning.jobs.list_events(JOB_ID, limit=20)

print(f"Recent training events for {JOB_ID}:\n")
for event in events.data:
    timestamp = datetime.fromtimestamp(event.created_at)
    print(f"[{timestamp}] {event.message}")

print(f"\n💡 Full event history available in Azure AI Foundry portal")

## 8. Next Steps

Once the RFT job completes (status = "succeeded"):

### Deploy the Fine-Tuned Model

1. Get the fine-tuned model ID from `job_status.fine_tuned_model`
2. Create a new agent deployment pointing to this model
3. Use the same agent code and tools as the baseline o4-mini agent

### Run Post-RFT Evaluation

1. Open `phase2_base_evaluations.ipynb`
2. Update the agent list to include your RFT-trained model
3. Run the evaluation on the same 62 validation scenarios
4. Compare results: baseline o4-mini vs RFT o4-mini

### Expected Improvements

- **Overall pass rate:** +15-25% improvement
- **Decision correctness:** Significant gain (policy application)
- **Financial accuracy:** Moderate gain (restocking fees)
- **Format compliance:** Slight gain (consistent output)

### Documentation

- See `docs/phase3_rft_training.md` for complete Phase 3 walkthrough
- See `DEMO_PLAN.md` for full 4-phase demo plan